<a href="https://colab.research.google.com/github/izzat-ai/learning-ai/blob/main/pandas/projects/MedCare_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Toshkentdagi xususiy klinika bizni yolladi. Ularning 3 yillik bemorlar ma'lumotlari bor, lekin hech qanday tahlil qilinmagan. Bizning vazifamiz — bu ma'lumotlardan amaliy biznes qarorlar chiqarish**

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

- *Dataset sun'iy intellektdan olindi*

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/AI learning/pandas_2026/projects/medcare_dataset.csv")
df.head()

,bemor_id,ism,yosh,jinsi,viloyat,bo_lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,narx_sum,tolov_turi,holat,qayta_murojaat,reyting
0,P00001,Bemor_1,52,Ayol,Farg'ona,Nevrologiya,Dr. Rahimov S,Migren,2022-02-20,14.0,2999095.0,Karta,Og'ir,True,1.0
1,P00002,Bemor_2,15,Erkak,Surxondaryo,Kardiologiya,Dr. Toshmatov B,Yurak yetishmovchiligi,2024-04-02,9.0,1465502.0,Naqd,Surunkali,True,3.0
2,P00003,Bemor_3,72,Erkak,Surxondaryo,Pediatriya,Dr. Tursunova G,Ich ketish,2023-03-28,23.0,4530088.0,Naqd,Surunkali,False,3.0
3,P00004,Bemor_4,61,Ayol,Qashqadaryo,Jarrohlik,Dr. Abdullayev K,Churraq,2024-07-02,6.0,1349110.0,Sug'urta,Yaxshilandi,True,4.0
4,P00005,Bemor_5,21,Erkak,Farg'ona,Jarrohlik,Dr. Abdullayev K,O'tkir appenditsit,2024-01-02,7.0,996462.0,Naqd,Yaxshilandi,False,3.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2015 entries, 0 to 2014
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   bemor_id        2015 non-null   object 
 1   ism             2015 non-null   object 
 2   yosh            2015 non-null   int64  
 3   jinsi           2015 non-null   object 
 4   viloyat         1990 non-null   object 
 5   bo_lim          2015 non-null   object 
 6   shifokor        2015 non-null   object 
 7   tashxis         2015 non-null   object 
 8   qabul_sanasi    2015 non-null   object 
 9   yotish_kunlari  1955 non-null   float64
 10  narx_sum        1985 non-null   float64
 11  tolov_turi      2015 non-null   object 
 12  holat           2015 non-null   object 
 13  qayta_murojaat  2015 non-null   bool   
 14  reyting         1975 non-null   float64
dtypes: bool(1), float64(3), int64(1), object(10)
memory usage: 222.5+ KB


- `qabul_sanasi` ustuni str formatida ekan , uni datetime qilish kerak
- `qayta_murojaat` booleanda berilgan , uni int qilishimiz kerak

In [ ]:
df.shape

(2015, 15)

In [ ]:
df.describe()

,yosh,yotish_kunlari,narx_sum,reyting
count,2015.000000,1955.000000,1.985000e+03,1975.000000
mean,41.184615,14.325831,2.533063e+06,3.852152
std,23.978902,8.649002,1.392165e+06,1.159697
min,1.000000,0.000000,1.500370e+05,1.000000
25%,20.000000,7.000000,1.340064e+06,3.000000
50%,40.000000,14.000000,2.493145e+06,4.000000
75%,62.000000,22.000000,3.731877e+06,5.000000
max,84.000000,29.000000,4.999666e+06,5.000000


- be'morlarning o'rtachasi 41 yosh ekan . Eng keksasi 84 yoshgacha bo'lsa , eng kichigi 1 yosh .
- be'morlar o'rtacha 14 kun davomida qolib davolanishgan . Ularning aksari (75% izi) 22 kun davomida yotishgan . Qolgan 25% esa bir hafta yotib davolanishgan .
- klinikada maksimal ravishda ko'p qolganlar - 5 milliongacha , o'rtacha esa 2.5 milliongacha to'lov qilishgan
- berilgan reytinglardan - be'morlarning klinikaga nisbatan bergan ballarini ko'rish mumkin . Be'morlarning 50% izi 4 reytingni bergan , bundan - ularning kamida yarmi klinikaga 4 va 5 baho bergan , bu yaxshi natija .

In [ ]:
df.isnull().sum()

,0
bemor_id,0
ism,0
yosh,0
jinsi,0
viloyat,25
bo_lim,0
shifokor,0
tashxis,0
qabul_sanasi,0
yotish_kunlari,60


- viloyat, yotish_kunlari, narx_sum va reyting ustunlarida tushib qolgan qiymatlar bor , ularni oldini olish kerak

In [ ]:
df.duplicated().sum()

np.int64(15)

- datasetimizda bir xil ma'lumotlardan bir nechta borligi aniq bo'ldi

In [ ]:
df['tashxis'].value_counts()

,count
tashxis,
Bronxit,110
Angina,87
Qalqonsimon bez,81
Travma,77
Osteoxondroz,76
Neyropatiya,74
Mioma,70
Ovarian kista,68
Semirishlik,67


- bemorlarda turli kasalliklar qayd etilgan . Buni tashhis natijalaridan ham bilib olsak bo'ladi

In [ ]:
df['jinsi'].value_counts()

,count
jinsi,
Ayol,1065
Erkak,950


In [ ]:
df['jinsi'].isnull().sum()

np.int64(0)

In [ ]:
df['bo_lim'].value_counts()

,count
bo_lim,
Endokrinologiya,270
Pediatriya,263
Nevrologiya,260
Ginekologiya,256
Ortopediya,253
Jarrohlik,246
Terapiya,241
Kardiologiya,226


In [ ]:
# bo_lim ustunini nomini o'zgartirish
df = df.rename(columns={'bo_lim':"bo'lim"})
df.columns

Index(['bemor_id', 'ism', 'yosh', 'jinsi', 'viloyat', 'bo'lim', 'shifokor',
       'tashxis', 'qabul_sanasi', 'yotish_kunlari', 'narx_sum', 'tolov_turi',
       'holat', 'qayta_murojaat', 'reyting'],
      dtype='object')

In [ ]:
df.describe(include='object')

,bemor_id,ism,jinsi,viloyat,bo'lim,shifokor,tashxis,qabul_sanasi,tolov_turi,holat
count,2015,2015,2015,1990,2015,2015,2015,2015,2015,2015
unique,2000,2000,2,10,8,18,31,926,4,4
top,P01888,Bemor_1888,Ayol,Navoiy,Endokrinologiya,Dr. Tursunova G,Bronxit,2023-07-14,Karta,Yaxshilandi
freq,2,2,1065,224,270,143,110,7,835,1187


- ushbu matnli ma'lumotlardan : viloyat ustunida bir qancha qiymatlar yoq'ligi ma'lum bo'lyapti
- bemor_id va ism ustunlarida 2000 ta ma'lumotlar takrorlanmas ma'lumotlardir
- idsi `P01888` ma'lumoti , qolganlarnikidan ko'p takrorlangan

#### 2. **Ma'lumotlarni tozalash**

In [ ]:
df.shape

(2015, 15)

In [ ]:
df.duplicated().sum()

np.int64(15)

- datasetimizda jami 15 ta duplikatlar bor ekan , ularni o'chiramiz

In [ ]:
# takroriy ma'lumotlarni o'chirish
df = df.drop_duplicates()
df.shape

(2000, 15)

In [ ]:
# null qiymatlarni tekshirish
df.isnull().sum()

,0
bemor_id,0
ism,0
yosh,0
jinsi,0
viloyat,25
bo'lim,0
shifokor,0
tashxis,0
qabul_sanasi,0
yotish_kunlari,60


In [ ]:
df['viloyat'].value_counts()

,count
viloyat,
Navoiy,221
Qashqadaryo,220
Farg'ona,215
Samarqand,197
Andijon,193
Toshkent,191
Xorazm,190
Surxondaryo,189
Buxoro,186


In [ ]:
# qaysi viloyatdanligi yozilmay qolganlarni "nomalum" bilan to'ldirish
df['viloyat'] = df['viloyat'].fillna("Noma'lum")
df['viloyat'].isnull().sum()

np.int64(0)

- yotish kunlari ustunida esa 60 ta ma'lumot yo'q . Ya'ni ko'p be'morlarni necha kun yotib davolangani noma'lum .
- yo'q qiymatlarni tashxis ustuniga qarab , uning o'rtacha qiymati bilan to'ldiramiz

In [ ]:
df['yotish_kunlari'] = df.groupby('tashxis')['yotish_kunlari'].transform(lambda x: x.fillna(x.median()))
df['yotish_kunlari'].isnull().sum()

np.int64(0)

- `narx_sum` ustunida 30 ta ma'lumotlar yo'q , ya'ni 30 ta bemorning qanchaga davolangani yozilmagan .
- davolanish narxi qancha bo'lgani - be'morning qaysi bo'limda davolanganiga yoki qaysi shifokor ko'rigidan o'tganiga bog'liq bo'ladi . Shuning uchun bo'limga qarab o'rtacha qiymat bilan to'ldiramiz

In [ ]:
df['narx_sum'] = df.groupby("bo'lim")['narx_sum'].transform(lambda x: x.fillna(x.mean()))
df['narx_sum'].isnull().sum()

np.int64(0)

In [ ]:
df.head()

,bemor_id,ism,yosh,jinsi,viloyat,bo'lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,narx_sum,tolov_turi,holat,qayta_murojaat,reyting
0,P00001,Bemor_1,52,Ayol,Farg'ona,Nevrologiya,Dr. Rahimov S,Migren,2022-02-20,14.0,2999095.0,Karta,Og'ir,True,1.0
1,P00002,Bemor_2,15,Erkak,Surxondaryo,Kardiologiya,Dr. Toshmatov B,Yurak yetishmovchiligi,2024-04-02,9.0,1465502.0,Naqd,Surunkali,True,3.0
2,P00003,Bemor_3,72,Erkak,Surxondaryo,Pediatriya,Dr. Tursunova G,Ich ketish,2023-03-28,23.0,4530088.0,Naqd,Surunkali,False,3.0
3,P00004,Bemor_4,61,Ayol,Qashqadaryo,Jarrohlik,Dr. Abdullayev K,Churraq,2024-07-02,6.0,1349110.0,Sug'urta,Yaxshilandi,True,4.0
4,P00005,Bemor_5,21,Erkak,Farg'ona,Jarrohlik,Dr. Abdullayev K,O'tkir appenditsit,2024-01-02,7.0,996462.0,Naqd,Yaxshilandi,False,3.0


- aksar be'morlar klinikaga yoki shifokorga ball bermagan . Lekin yaxshi doktrlarga ko'pincha yuqori ball berishadi . Shundan kelib chiqib har bir doktrga berilgan o'rtacha ball bilan NaN larni to'ldiramiz

In [ ]:
df['reyting'] = df.groupby('shifokor')['reyting'].transform(lambda x: x.fillna(x.mean()))
df['reyting'].isnull().sum()

np.int64(0)

In [ ]:
df.isnull().sum()

,0
bemor_id,0
ism,0
yosh,0
jinsi,0
viloyat,0
bo'lim,0
shifokor,0
tashxis,0
qabul_sanasi,0
yotish_kunlari,0


#####**Ma'lumot turlarini o'zgartirish**

In [ ]:
df['qabul_sanasi'] = pd.to_datetime(df['qabul_sanasi'])
df['qabul_sanasi'].dtype

dtype('<M8[ns]')

- `qayta_murojaat` ustuni booleanda berilgan . Uni int8 turiga o'zgartiramiz . Shunda True lar 1 bo'ladi , False lar esa 0 bo'ladi
- bundan qayta_murojaat=1 bo'lsa , be'mor klinikaga avval ham kelgan bo'ladi , agarda 0 ga teng bo'lsa aksincha

In [ ]:
df['qayta_murojaat'] = df['qayta_murojaat'].astype(np.int8)
df['qayta_murojaat'].head()

,qayta_murojaat
0,1
1,1
2,0
3,1
4,0


In [ ]:
# takrorlangan qatorlar qolmaganini tekshirish
df.duplicated().sum()

np.int64(0)

In [ ]:
df.shape

(2000, 15)

In [ ]:
# qayta indekslab qo'yish
df = df.reset_index(drop=True)

In [ ]:
df.head()

,bemor_id,ism,yosh,jinsi,viloyat,bo'lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,narx_sum,tolov_turi,holat,qayta_murojaat,reyting
0,P00001,Bemor_1,52,Ayol,Farg'ona,Nevrologiya,Dr. Rahimov S,Migren,2022-02-20,14.0,2999095.0,Karta,Og'ir,1,1.0
1,P00002,Bemor_2,15,Erkak,Surxondaryo,Kardiologiya,Dr. Toshmatov B,Yurak yetishmovchiligi,2024-04-02,9.0,1465502.0,Naqd,Surunkali,1,3.0
2,P00003,Bemor_3,72,Erkak,Surxondaryo,Pediatriya,Dr. Tursunova G,Ich ketish,2023-03-28,23.0,4530088.0,Naqd,Surunkali,0,3.0
3,P00004,Bemor_4,61,Ayol,Qashqadaryo,Jarrohlik,Dr. Abdullayev K,Churraq,2024-07-02,6.0,1349110.0,Sug'urta,Yaxshilandi,1,4.0
4,P00005,Bemor_5,21,Erkak,Farg'ona,Jarrohlik,Dr. Abdullayev K,O'tkir appenditsit,2024-01-02,7.0,996462.0,Naqd,Yaxshilandi,0,3.0


####3. **Ma'lumotlarni Filtrlash va O'zgartirish**

In [64]:
# Kardialogiya bo'limidagi yoshi 50 dan yuqori be'morlarni ma'lumotlarini ko'rish
df.loc[(df["bo'lim"]=='Kardiologiya')&(df['yosh']>50)]

,bemor_id,ism,yosh,jinsi,viloyat,bo'lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,narx_sum,tolov_turi,holat,qayta_murojaat,reyting
20,P00021,Bemor_21,76,Ayol,Navoiy,Kardiologiya,Dr. Yusupova M,Yurak yetishmovchiligi,2024-10-01,28.0,4255810.0,Karta,Yaxshilandi,1,3.0
35,P00036,Bemor_36,64,Erkak,Xorazm,Kardiologiya,Dr. Karimov A,Stenokardiya,2023-09-23,3.0,3525896.0,Karta,Surunkali,0,2.0
44,P00045,Bemor_45,60,Erkak,Qashqadaryo,Kardiologiya,Dr. Yusupova M,Aritmiya,2024-05-18,4.0,632450.0,Naqd,Yaxshilandi,0,5.0
70,P00071,Bemor_71,74,Erkak,Buxoro,Kardiologiya,Dr. Karimov A,Aritmiya,2023-09-11,5.0,4541649.0,Naqd,Yaxshilandi,1,4.0
112,P00113,Bemor_113,62,Erkak,Navoiy,Kardiologiya,Dr. Toshmatov B,Aritmiya,2024-01-10,29.0,179092.0,Naqd,Og'ir,0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1915,P01916,Bemor_1916,54,Ayol,Navoiy,Kardiologiya,Dr. Toshmatov B,Stenokardiya,2024-07-23,18.0,2648002.0,Sug'urta,Yaxshilandi,0,5.0
1918,P01919,Bemor_1919,65,Ayol,Xorazm,Kardiologiya,Dr. Yusupova M,Gipertenziya,2024-03-11,5.0,1724255.0,Sug'urta,Yaxshilandi,0,5.0
1965,P01966,Bemor_1966,51,Ayol,Samarqand,Kardiologiya,Dr. Yusupova M,Stenokardiya,2023-11-01,27.0,2683922.0,Sug'urta,Yaxshilandi,0,5.0
1967,P01968,Bemor_1968,68,Erkak,Xorazm,Kardiologiya,Dr. Toshmatov B,Aritmiya,2024-09-19,27.0,235350.0,Naqd,Yaxshilandi,1,4.0


In [68]:
# 1-10 qatorli ma'lumotlarni va be'morlarni ismi, yoshi va bo'limini ko'rish
df.iloc[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [1, 2, 5]]

,ism,yosh,bo'lim
0,Bemor_1,52,Nevrologiya
1,Bemor_2,15,Kardiologiya
2,Bemor_3,72,Pediatriya
3,Bemor_4,61,Jarrohlik
4,Bemor_5,21,Jarrohlik
5,Bemor_6,83,Terapiya
6,Bemor_7,75,Nevrologiya
7,Bemor_8,75,Nevrologiya
8,Bemor_9,24,Ortopediya
9,Bemor_10,3,Kardiologiya


In [73]:
# narxi bir milliondan yuqori hamda holati yaxshi bo'lganlarni ma'lumotlarini ko'rish
df.query("narx_sum>1000000 and holat=='Yaxshilandi'")

,bemor_id,ism,yosh,jinsi,viloyat,bo'lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,narx_sum,tolov_turi,holat,qayta_murojaat,reyting
3,P00004,Bemor_4,61,Ayol,Qashqadaryo,Jarrohlik,Dr. Abdullayev K,Churraq,2024-07-02,6.0,1349110.0,Sug'urta,Yaxshilandi,1,4.0
10,P00011,Bemor_11,22,Ayol,Surxondaryo,Kardiologiya,Dr. Yusupova M,Aritmiya,2023-03-31,20.0,1652718.0,Sug'urta,Yaxshilandi,0,5.0
11,P00012,Bemor_12,53,Ayol,Surxondaryo,Nevrologiya,Dr. Holiqova N,Neyropatiya,2023-07-24,5.0,2978340.0,Karta,Yaxshilandi,1,2.0
12,P00013,Bemor_13,2,Ayol,Buxoro,Jarrohlik,Dr. Abdullayev K,Travma,2023-02-05,0.0,1344737.0,Naqd,Yaxshilandi,1,5.0
13,P00014,Bemor_14,30,Erkak,Qashqadaryo,Jarrohlik,Dr. Xasanov R,Travma,2022-02-16,3.0,3693525.0,Naqd,Yaxshilandi,0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1986,P01987,Bemor_1987,77,Erkak,Qashqadaryo,Pediatriya,Dr. Tursunova G,ARVI,2022-06-08,7.0,1800882.0,Naqd,Yaxshilandi,1,3.0
1987,P01988,Bemor_1988,4,Erkak,Navoiy,Kardiologiya,Dr. Toshmatov B,Aritmiya,2022-01-23,9.0,2609150.0,Naqd,Yaxshilandi,0,3.0
1988,P01989,Bemor_1989,75,Erkak,Navoiy,Terapiya,Dr. Mirzayev J,Gastrit,2023-07-14,21.0,2175216.0,Naqd,Yaxshilandi,1,3.0
1991,P01992,Bemor_1992,39,Erkak,Surxondaryo,Kardiologiya,Dr. Toshmatov B,Yurak yetishmovchiligi,2024-01-04,7.0,2981455.0,Naqd,Yaxshilandi,0,5.0


In [76]:
# naqt to'lagan va qayta murojat qilgan be'morlar ma'lumotlari
df.query("tolov_turi=='Naqd' and qayta_murojaat==1")

,bemor_id,ism,yosh,jinsi,viloyat,bo'lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,narx_sum,tolov_turi,holat,qayta_murojaat,reyting
1,P00002,Bemor_2,15,Erkak,Surxondaryo,Kardiologiya,Dr. Toshmatov B,Yurak yetishmovchiligi,2024-04-02,9.0,1465502.0,Naqd,Surunkali,1,3.0
8,P00009,Bemor_9,24,Ayol,Farg'ona,Ortopediya,Dr. Normatov U,Osteoxondroz,2023-02-04,15.0,1769814.0,Naqd,Og'ir,1,4.0
12,P00013,Bemor_13,2,Ayol,Buxoro,Jarrohlik,Dr. Abdullayev K,Travma,2023-02-05,0.0,1344737.0,Naqd,Yaxshilandi,1,5.0
14,P00015,Bemor_15,38,Erkak,Toshkent,Kardiologiya,Dr. Karimov A,Gipertenziya,2022-06-21,10.0,548825.0,Naqd,Yaxshilandi,1,3.0
30,P00031,Bemor_31,62,Erkak,Toshkent,Ortopediya,Dr. Botirov A,Sinish,2024-02-03,22.0,4734751.0,Naqd,Surunkali,1,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1978,P01979,Bemor_1979,35,Erkak,Navoiy,Ginekologiya,Dr. Ismoilova Z,Mioma,2022-02-06,9.0,705134.0,Naqd,Yaxshilandi,1,2.0
1979,P01980,Bemor_1980,8,Ayol,Navoiy,Ginekologiya,Dr. Ismoilova Z,Mioma,2022-07-25,27.0,4297036.0,Naqd,Yaxshilandi,1,4.0
1986,P01987,Bemor_1987,77,Erkak,Qashqadaryo,Pediatriya,Dr. Tursunova G,ARVI,2022-06-08,7.0,1800882.0,Naqd,Yaxshilandi,1,3.0
1988,P01989,Bemor_1989,75,Erkak,Navoiy,Terapiya,Dr. Mirzayev J,Gastrit,2023-07-14,21.0,2175216.0,Naqd,Yaxshilandi,1,3.0
